# Writing your own reader

ECGProcess already reads XML and DICOM. A recording kept in another
layout can go through the same tools, including `ECGTable`, once a
small reader exists for that layout.

The reader opens one file and puts every value into `raw_data`, a
flat dictionary that maps a tag name to a value. From there the
shared starting point, `BaseReader`, takes over. Its `extract`
method uses the configuration file to pull values out of that
dictionary, sort them into `MetaData`, `WaveForms`, `MedianBeats`,
and `OtherData`, and, when asked, resample the signals or fill in
extra limb leads.

What you still write is `__call__`, the method that runs when the
reader is used like a function. The cells below build a reader for
a simple CSV file in which each column is one lead.

In [1]:
import pathlib
import tempfile
from dataclasses import dataclass, field

import numpy as np
import pandas as pd

from ecgprocess.process import BaseReader
from ecgprocess.tabular import ECGTable
from ecgprocess.constants import CoreData as Core
from ecgprocess.errors import ReaderOutputError
from ecgprocess.utils.general import ManagedProperty
from ecgprocess.utils.config_tools import ConfigParser, DataMap
from ecgprocess.example_data.examples import config_file

CProc = Core.ProcessingData

## Example data

Two short CSV files are written here. Each holds two seconds of
leads I and II, sampled 250 times a second. The file name is also
the patient identifier.

In [2]:
data_dir = pathlib.Path(tempfile.mkdtemp())
time = np.linspace(0, 2, 500)
paths = []
for offset, patient in enumerate(['P001', 'P002']):
    path = data_dir / f'{patient}.csv'
    pd.DataFrame({
        'lead_I': np.sin(2 * np.pi * time) + offset,
        'lead_II': np.cos(2 * np.pi * time) + offset,
    }).to_csv(path, index=False)
    paths.append(str(path))
pd.read_csv(paths[0]).head(3)

,lead_I,lead_II
0,0.000000,1.000000
1,0.025180,0.999683
2,0.050345,0.998732


## The reader

`CSVReader` is a dataclass, a class that lists its settings as
fields, and it builds on `BaseReader`. The CSV columns supply the
leads. Three further values are added because the file does not
contain them: the patient identifier, the sampling frequency (a
setting on the reader), and how many samples the file holds.

`set_with_setter` writes the dictionary into `raw_data`, which is
how the built-in readers store it. `__call__` hands the reader back.

In [3]:
@dataclass
class CSVReader(BaseReader):
    """Open a CSV file that stores one lead in each column."""
    raw_data = ManagedProperty(CProc.RAW, dict)
    sampling_rate: int = field(default=250)

    def __call__(self, path, verbose=False, **kwargs):
        table = pd.read_csv(path)
        flat = {col: table[col].to_list() for col in table.columns}
        flat['patient'] = pathlib.Path(path).stem
        flat['rate'] = self.sampling_rate
        flat['n_samples'] = len(table)
        # raw_data stays locked, so the managed setter writes the map.
        getattr(type(self), CProc.RAW).set_with_setter(self, flat)
        return self

## The configuration file

Each line maps an ECGProcess name on the left to a key in the flat
dictionary on the right. These files have no median beats, so there
is no `MedianBeats` section.

In [4]:
config_path = str(data_dir / 'config.txt')
_ = config_file(path=config_path, text={
    'MetaData': [
        'unique identifier\tpatient',
        'sampling frequency (original)\trate',
        'sampling number (waveforms)\tn_samples',
    ],
    'WaveForms': ['I\tlead_I', 'II\tlead_II'],
})
parser = ConfigParser(config_path)()
parser.map(mapper=DataMap())

## Using the reader

Call the reader the same way as a built-in reader. `extract`,
inherited from `BaseReader`, resampled the waveforms from 250 to
500 Hz, so each lead now has 1000 samples. These files have no
median-beat samples, so that category is left as it is.

In [5]:
reader = CSVReader()(paths[0]).extract(config=parser)
print(reader.MetaData['unique identifier'])
print({lead: signal.shape for lead, signal in reader.WaveForms.items()
       if signal is not None})

P001
{'I': (1000,), 'II': (1000,)}


The same reader can be handed to `ECGTable` to process several
files. These files have no median beats, so that category is turned
off with `extract_median=False`.

In [6]:
table = ECGTable(CSVReader(), path_list=paths, extract_median=False)
table = table().get_table(parsed_config=parser)
table.WaveForms.dropna().head()

,key,Sampling sequence,Lead,Voltage,Signal type
0,P001,0,I,8.881784e-19,WaveForms
1,P001,0,II,1.000000e+00,WaveForms
12,P001,1,I,9.150674e-03,WaveForms
13,P001,1,II,9.999064e-01,WaveForms
24,P001,2,I,2.518045e-02,WaveForms


## What can go wrong

`BaseReader` applies two checks.

A reader with no `__call__` method cannot be created:

In [7]:
@dataclass
class NoCallReader(BaseReader):
    pass

try:
    NoCallReader()
except TypeError as error:
    print(error)

Can't instantiate abstract class NoCallReader with abstract method __call__


`ECGTable` also checks that the reader filled every data category
it is about to read. The check matters most when `extract` is
replaced: the replacement should call `super().extract(...)`, or
set every category itself. `ForgetfulReader` sets only `MetaData`.
`ECGTable` then stops with `ReaderOutputError` rather than treating
each file as empty.

In [8]:
@dataclass
class ForgetfulReader(CSVReader):
    def extract(self, config, **kwargs):
        self.MetaData = {'unique identifier': self.raw_data['patient']}
        return self

try:
    ECGTable(ForgetfulReader(), path_list=paths,
             extract_median=False)().get_table(parsed_config=parser)
except ReaderOutputError as error:
    print(error)

`ForgetfulReader` did not set `WaveForms` after `extract`. Either populate it in the reader, or set the matching extract_meta/extract_wave/extract_median flag to `False` on ECGTable if the reader does not provide this data.


A category switched off on `ECGTable` (`extract_meta`,
`extract_wave`, or `extract_median` set to `False`) is not checked,
and the reader may leave it unset.

If an XML schema is passed to `ECGTable`, it is forwarded to the
reader as `schema=`. Only a reader that accepts that argument can
be used with a schema.